# VectorLess RAG

# Vectorless RAG vs. Traditional RAG

### 1. What is Vectorless RAG?
**Vectorless RAG** (Retrieval-Augmented Generation) is an alternative retrieval architecture that eliminates the reliance on traditional vector embeddings and vector similarity search. Instead, it leverages the **logical structure** of documents (such as hierarchical trees, tables of contents, document layouts, indexes, or page-level metadata) or direct keyword-based search mechanisms (like BM25, TF-IDF, or structured relational queries) to locate and retrieve context.

In structure-based/reasoning-based Vectorless RAG:
- The system treats documents as hierarchical entities (e.g., sections, subsections, tables, pages).
- It uses LLM-guided routing/reasoning to navigate the document structure or consults a pre-built index of key terms, entities, and summaries mapped to exact pages/sections.
- It retrieves the precise structural block (e.g., a specific page or table cell) instead of a fixed-size text chunk.

---

### 2. How it Differs from Traditional RAG
- **Traditional RAG**: Converts raw text chunks into high-dimensional numerical vectors using an embedding model. Retrieval is done by calculating semantic similarity (e.g., Cosine Similarity) between the user query embedding and the chunk embeddings.
- **Vectorless RAG**: Works directly with raw text, structured page layouts, hierarchical indices, or relational/graph structures. Retrieval is driven by metadata routing, structural traversal, or lexical indices (like BM25) to map the search to specific, deterministic document components.

---

### 3. Comparison Matrix

| Feature | Traditional (Vector-based) RAG | Vectorless RAG |
| :--- | :--- | :--- |
| **Retrieval Mechanism** | Semantic Similarity (mathematical closeness of embeddings). | Logical structure navigation, lexical indices (BM25), or LLM routing. |
| **Data Representation** | Flat list of vector embeddings representing text chunks. | Hierarchical trees, page/metadata indexes, or structured tables. |
| **Search Paradigm** | Fuzzy, semantic, conceptual search. | Exact, structured, keyword, or document-navigation search. |
| **Chunking** | Typically fixed-size (e.g., 512 tokens) with overlap. | Page-based, section-based, or document-hierarchy-based. |
| **Explainability** | Low (hard to explain why a specific chunk has a high cosine score). | High (follows an audit trail of pages, sections, or keywords). |
| **Setup Cost** | Needs embedding models and specialized Vector DBs. | Needs document parsing, index builders, or standard databases. |

---

### 4. Step-by-Step Real-time Flow Examples

Let's trace how both systems handle a query against a **100-page financial quarterly PDF report**.
**User Query:** *"What was the total revenue in Q3?"*

#### **Scenario A: Traditional (Vector-Based) RAG Flow**

![Traditional RAG Flow](./images/traditional_rag_flow.png)

1. **Offline Ingestion & Chunking**:
   - The PDF document is parsed and split into arbitrary fixed-size text chunks (e.g., 500 characters with 50-character overlap).
   - Each text chunk is sent to an embedding model (e.g., OpenAI's `text-embedding-3-small`) to generate a 1536-dimensional vector.
   - These vectors (and their associated text) are indexed and saved inside a Vector Database (e.g., Pinecone, Chroma).
2. **Query Vectorization**:
   - The user asks: *"What was the total revenue in Q3?"*
   - The query is converted into a query vector using the same embedding model.
 3. **Similarity Search**:
   - The system performs a Cosine Similarity search in the Vector Database.
   - It retrieves the top 3 chunks with the highest similarity score.
   - *Potential Issue:* If the financial tables are broken up across chunk boundaries, the numbers of Q3 revenue might fall in Chunk #42, but the headers and column labels might reside in Chunk #41. If only Chunk #42 is retrieved because of raw keyword proximity, the context is incomplete.
4. **Generation**:
   - The system builds a prompt combining the retrieved chunks as context and passes it to the LLM.
   - The LLM attempts to answer. If the table formatting was broken, it might hallucinate or report that the information is missing.

#### **Scenario B: Vectorless RAG Flow (e.g., Page-Index Approach)**

![Vectorless RAG Flow](./images/vectorless_rag_flow.png)

1. **Offline Structuring & Key/Value Indexing**:
   - The system parses the document while retaining page boundaries, layouts, and tables intact.
   - A lightweight indexer (often using an LLM or keyword extraction parser) analyzes each page and outputs a structured key-value page index describing what is on that page.
   - *Example index entry:* `{"Page 15": "Contains Q3 Consolidated Statements of Operations: Total Revenue, Gross Margin, Net Income table"}`
   - This structured mapping (Index -> Page Number) is stored in a simple, fast key-value store or relational DB.
2. **Logical Query Routing**:
   - The user asks: *"What was the total revenue in Q3?"*
   - A routing mechanism (like a lightweight LLM router or standard lexical/BM25 search over the page summaries) is executed.
   - The router queries the page metadata index and immediately returns that **Page 15** is the exact location of the Q3 Consolidated Statement of Operations.
3. **Clean Layout Extraction**:
   - The system retrieves the entire, raw contents of **Page 15** in its original structural integrity (preserving rows, columns, and headers).
4. **Generation**:
   - The entire page's context is injected into the prompt.
   - The LLM processes the full, unbroken table, retrieves the correct Q3 revenue figure, and outputs the answer, citing exactly: *"Total revenue in Q3 was $X billion (Page 15)."*

---

### 5. Advantages & Disadvantages

#### **Traditional RAG**
*   **Advantages:**
    *   Excellent at matching synonyms and understanding user intent/vague queries.
    *   Works well with massive, unstructured collections of diverse text files.
    *   Fast retrieval once the vector database index is built.
*   **Disadvantages:**
    *   **Lost in Chunking**: Splitting files at arbitrary token sizes can break table structures or separate context (e.g., an item is on page 5, but its column headers are on page 4).
    *   **Out-of-domain embeddings**: Embedding models might fail to capture highly technical domain-specific terms or codes.
    *   **High DB Overhead**: Maintaining vector indexes can be resource-intensive.

#### **Vectorless RAG**
*   **Advantages:**
    *   **Ultra-precise**: Retrieves exact pages or tables without losing formatting, headers, or structural hierarchy.
    *   **No embedding mismatch**: Ideal for domain-specific jargon (e.g., codes, legal clauses) where exact matches matter.
    *   **Traceability**: Easily provides reference citations (e.g., "See Section 4.2, Page 12").
*   **Disadvantages:**
    *   **Strict formatting requirements**: Relies on documents having a well-organized layout or clear structure.
    *   **Fuzzy queries struggle**: If a user queries "motor insurance" and the document says "automobile coverage", a pure vectorless system without a semantic layer might fail to retrieve it.
    *   **Higher query-time LLM cost**: If utilizing LLMs to navigate document trees, it adds latency and API call costs.

---

### 6. Production-Grade Usage Guidelines

*   **Choose Traditional RAG when:**
    *   You have huge, flat datasets (e.g., millions of customer support transcripts).
    *   Users write conversational, natural language questions without technical terms.
    *   You need rapid, low-latency semantic search across a vast catalog.

*   **Choose Vectorless RAG when:**
    *   You work with structured manuals, technical documentation, financial sheets, or legal contracts.
    *   Deterministic retrieval is critical (e.g., answering "What is the fee in section 2.b?").
    *   You want to avoid the cost and complexity of setting up and updating vector databases.

*   **Hybrid RAG (Best of Both Worlds):**
    *   In enterprise setups, the gold standard is often combining the two: using vector search for initial broad recall and vectorless index/structure lookup to filter, sort, and pin down the exact paragraphs or cells.


## Installation & Setup
1. PageIndex API Key: https://developer.pageindex.ai/overview
    - It is a Vectorless RAG service
    - Generate the API key and place it in .env file
2. OpenAI or Groq or any other LLM API Key is needed
    - We are using Groq here, please the API KEY(GROQ_API_KEY) in .env
3. Packages to Install
    - pageindex, openai or groq

In [ ]:
import os, json, time
from dotenv import load_dotenv
load_dotenv()

PAGEINDEX_API_KEY = os.environ.get('PAGEINDEX_API_KEY')
GROQ_API_KEY = os.environ.get('GROQ_API_KEY')

In [8]:
from pageindex import PageIndexClient
from langchain_groq import ChatGroq
llm = ChatGroq(groq_api_key=GROQ_API_KEY, model_name='llama-3.1-8b-instant', temperature=0.1, max_tokens=1024)

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

## Upload & Index a PDF
- What happens here:
1. Upload your PDF to the PageIndex cloud
2. PageIndex uses an LLM to read the document structure
3. Builds a hierarchical tree index (like a smart Table of Contents)
4. Returns a doc_id for all future operations
- Why NO chunking?
    - Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.

In [10]:
PDF_PATH = "./pdfs/RAGvsNodeBasedSearchPaper.pdf"

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

📤 Uploading: ./pdfs/RAGvsNodeBasedSearchPaper.pdf
✅ Uploaded!
📋 Document ID: pi-cmspzgh1j019501qtk2xrfxzu
   (Save this ID — you'll use it throughout the notebook)


In [11]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: completed

✅ Tree index ready!


## Inspect the Tree Structure
What the tree looks like:
```
Document
├── Introduction (pages 1-3)
│   └── Background (pages 1-2)
├── Financial Stability (pages 21-31)
│   ├── Monitoring Vulnerabilities (pages 22-28)
│   └── International Cooperation (pages 28-31)
└── Conclusion (pages 45-47)
```
- Each node has:
    - node_id — unique ID used during retrieval
    - title — section heading
    - page_index — page number in original PDF
    - text — section summary (when node_summary=True)
    - nodes — child sections (nested)
- This structure is what the LLM reasons over during retrieval.

In [12]:

# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "Rethinking Retrieval: From Traditional Retrieval Augmented Generation to Agentic and Non-Vector Reasoning Systems in the Financial Domain for Large Language Models",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "This paper presents a systematic evaluation of RAG architectures for financial Q&A, comparing vector-based agentic systems against hierarchical node-based approaches. Using a benchmark of SEC filings, the study demonstrates that vector-based agentic RAG, enhanced by cross-encoder reranking and small-to-big chunk retrieval, provides superior retrieval accuracy and answer quality, while highlighting the critical cost-performance tradeoffs required for production deployment.",
  "text": "# Rethinking Retrieval: From Traditional Retrieval Augmented Generation to Agentic and Non-Vector Reasoning Systems in the Financial Domain for Large Language Models\n\nElias Lumer, Matt Melich, Olivia Zino, Elena Kim, 

In [ ]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Rethinking Retrieval: From Traditional Retrieval Augmented Generation to Agentic and Non-Vector Reasoning Systems in the Financial Domain for Large Language Models  (p.1)
  └─ [0001] 1 INTRODUCTION  (p.1)
  └─ [0002] 2 RELATED WORK  (p.2)
  └─ [0003] 3 METHOD  (p.3)
    └─ [0004] 3.1 Dataset Construction and Document Processing  (p.3)
    └─ [0005] 3.2 Retrieval System Architectures  (p.4)
    └─ [0006] 3.3 Advanced RAG Enhancement Techniques  (p.4)
    └─ [0007] 3.4 Evaluation Framework  (p.4)
  └─ [0008] 4 EXPERIMENTS  (p.5)
    └─ [0009] 4.1 Experimental Settings  (p.5)
    └─ [0010] 4.2 Preprocessing and Node Generation  (p.5)
    └─ [0011] 4.3 Retrieval Architecture Comparison  (p.5)
    └─ [0012] 4.4 Cross-Encoder Reranking Evaluation  (p.6)
    └─ [0013] 4.5 Small-to-Big Retrieval Evaluation  (p.6)
  └─ [0014] 5 DISCUSSION  (p.6)
    └─ [0015] 5.1 Vector-Based and Hierarchical Reasoning Systems  (p.6)
    └─ [0016] 5.2 Cross-Encoder Reranking  

In [ ]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 20
   Each node = one retrievable section of the document


In [23]:
pageindex_tree[0].keys()

dict_keys(['title', 'node_id', 'page_index', 'prefix_summary', 'text', 'nodes'])

## LLM Tree Search — The Core of PageIndex
- __This is where PageIndex fundamentally differs from vector RAG.__
- __Vector RAG retrieval:__
    -  ```query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks```

    - Problem: finds what's similar, not what's relevant

- __PageIndex retrieval:__
    - ```query + tree → LLM reasons → "node 0007 and 0008 contain the answer"```
    - Advantage: LLM understands document structure, context, and intent

The LLM acts like a human expert scanning a Table of Contents.

In [63]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────
import re
def llm_tree_search(query: str, tree: list, model: str) -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
        Your task: identify which node IDs most likely contain the answer to the query.
        Think step-by-step about which sections are relevant.

        Query: {query}

        Document Tree:
        {json.dumps(compressed_tree, indent=2)}

        Reply ONLY in this exact JSON format:
        {{
            "thinking": "<your step-by-step reasoning>",
            "node_list": ["node_id1", "node_id2"]
        }}
    """

    response = model.invoke(prompt)
    content = response.content.strip()
    
    # Clean markdown formatting if present
    if content.startswith("```"):
        content = re.sub(r"^```(?:json)?\n?", "", content)
        content = re.sub(r"\n?```$", "", content)
        content = content.strip()
        
    try:
        parsed_response = json.loads(content, strict=False)
    except json.JSONDecodeError as e:
        # Fallback to try and extract json substring if LLM returned extra text
        match = re.search(r"\{.*\}", content, re.DOTALL)
        if match:
            parsed_response = json.loads(match.group(0), strict=False)
        else:
            raise e
            
    return parsed_response


In [47]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "WHat is RAG Compared with in this paper is it with Non vector based approach???"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree, llm)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: WHat is RAG Compared with in this paper is it with Non vector based approach???

🧠 LLM Reasoning:
To identify which node IDs most likely contain the answer to the query, I will follow these steps:

1. Identify the main keywords in the query: 'RAG', 'Compared with', 'Non-vector based approach'.
2. Look for nodes that contain these keywords in their title or summary.
3. Since the query is asking for a comparison between RAG and a non-vector based approach, I will focus on nodes that discuss RAG and its comparison with other methods.
4. I will also consider nodes that discuss retrieval systems and architectures, as they may provide information on how RAG is compared with other approaches.

Based on these steps, I will identify the node IDs that are most likely to contain the answer to the query.

🎯 Selected Node IDs: ['0002', '0011', '0012']


## Full End-to-End RAG Pipeline
- 3 steps:
    1. Tree Search → LLM picks relevant node_ids
    2. Retrieve → Fetch the actual section content from those nodes
    3. Generate → LLM writes a grounded answer with page citations
- What makes this better than vector RAG:
    - Retrieved content has titles + page numbers (traceable)
    - LLM can cite exactly which section the answer comes from
    - No hallucination from irrelevant chunks

In [53]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [54]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str) -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = model.invoke(prompt)
    content = response.content.strip()
    
    return content


In [55]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, model, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree, model)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes, model)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [56]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="WHat is RAG Compared with in this paper is it with Non vector based approach???",
    tree=pageindex_tree,
    model=llm
)

🔍 Query: WHat is RAG Compared with in this paper is it with Non vector based approach???

🧠 Reasoning: To identify which node IDs most likely contain the answer to the query, I will follow these steps:

1. Identify the main keywords in the query: 'RAG', 'Compared with', 'Non-vector based approach'.
2. ...
🎯 Retrieved node IDs: ['0002', '0011', '0012']
📄 Sections found: ['2 RELATED WORK', '4.3 Retrieval Architecture Comparison', '4.4 Cross-Encoder Reranking Evaluation']

📝 Answer:
RAG is compared with a non-vector based approach, specifically a hierarchical node-based system, in this paper (Section: '4.3 Retrieval Architecture Comparison' | Page 5). The hierarchical node-based system uses structured representations and reasoning-based retrieval without vector embeddings (Section: '2.3 Non-Vector and Structured Retrieval Approaches' | Page 3).


In [58]:
# ── Test with multiple queries ───────────────────────────────────────────────
test_queries = [
    "How is it proved in the paper that RAG is effecient?",
    "What is the summary of this RAG paper?",
    "What all topics where discussed in the RAG Paper?",
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, llm, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)


Q: How is it proved in the paper that RAG is effecient?
A: The efficiency of RAG is proved by comparing the vector-based agentic RAG system with the hierarchical node-based system, where the vector-based system achieved a 68% win rate with faster latency (5.2 vs 5.98 seconds) (4.3 Retrieval Architecture Comparison, Page 5)....
-------------------------------------------------------

Q: What is the summary of this RAG paper?
A: This RAG paper presents a systematic evaluation comparing vector-based agentic RAG against hierarchical node-based reasoning systems for financial document question answering. The paper finds that vector-based agentic RAG achieves a 68% win rate over hierarchical node-based systems with comparable l...
-------------------------------------------------------

Q: What all topics where discussed in the RAG Paper?
A: The RAG Paper discussed the following topics:

1. Retrieval-Augmented Generation (RAG) and its limitations (2.1 Retrieval-Augmented Generation, p. 2)
2

---
## 🎓Expert-Guided Retrieval

**The killer feature no one talks about.**

With vector RAG, injecting domain expertise requires **fine-tuning the embedding model** — expensive and time-consuming.

With PageIndex, you just **add rules to the prompt**:

```
"If the query mentions EBITDA → prioritize the MD&A section"
"If the query is about risks  → check Part I, Item 1A"
```

This makes PageIndex instantly adaptable to any domain — finance, legal, medical, technical — without any model training.

- Define domain expert rules
    - These are routing rules that tell the LLM WHERE to look for specific queries.
    - Think of it as encoding a senior analyst's institutional knowledge.



In [59]:
# ── Expert Routing Rules — Advanced Route of Learning AI ─────────────────────
# 21 Modules | 38 Sections | 481 Topics
FINANCIAL_EXPERT_RULES = """
Route queries to the correct module using these rules:
 
M1  Neural Network Refresher   → backprop, activations, optimizers, PyTorch basics
M2  Hardware                   → GPU, TPU, Apple Silicon, compute infrastructure
M3  Transformers 101           → attention, self-attention, encoder-decoder, MHA
M4  Tokenization               → BPE, WordPiece, SentencePiece, Byte Latent Transformers
M5  Finetuning Architectures   → hands-on BERT/GPT/T5 finetuning, Hugging Face
M6  KV Cache & Attention       → KV cache, Flash Attention, MQA, GQA, RoPE, vLLM
M7  Scaling Laws               → Kaplan, Chinchilla, compute-optimal training
M8  Mixture of Experts         → MoE, sparse computation, Mixture of Depths
M9  Modern LLM Finetuning      → LoRA, QLoRA, SFT, DPO, PPO, RLHF, GRPO, ORPO,
                                  quantization, TRL, Unsloth, synthetic data,
                                  reasoning models, evaluation, deployment
M10 SLM                        → small language models, pruning, when SLM vs LLM
M11 Knowledge Distillation     → student-teacher, soft labels, DistilBERT, DeepSeek-R1
M12 Hybrid Architectures       → Mamba, RWKV, SSMs, Jamba, Nemotron, beyond Transformers
M13 Vision Foundations         → ViT, patch embeddings, CLIP, SigLIP, DINOv2
M14 Visual Language Models     → VLM architecture, aligner, multimodal reasoning
M15 Stable Diffusion & DiT     → DDPM, latent diffusion, FLUX.1, ControlNet, DreamBooth
M16 Embedding Models           → dense, sparse, binary, Matryoshka, MRL, fine-tuning
M17 RAG                        → chunking, BM25, ColBERT, hybrid RAG, rerankers,
                                  self/corrective/adaptive/agentic RAG, Graph RAG,
                                  multi-modal RAG, ColPali, RAG security
M18 Context Engineering        → prompt vs context engineering, memory architecture,
                                  context compression, KV cache, agent context lifecycle
M19 DSPy                       → signatures, modules, MIPROv2, self-optimizing RAG
M20 Agents                     → ReAct, MCP, LangGraph, CrewAI, browser agents,
                                  A2A, guardrails, observability, evaluation
M21 RL                         → PPO, GRPO, DAPO, GSPO, CISPO, reward models,
                                  RLHF vs RLVR, policy gradient, DeepSeek-R1 training
 
Cross-cutting rules:
- "learning path / where to start"     → M1 → M2 → M3 in order
- "production / deployment / serving"  → M9 (quantization) + M20 (agents)
- "fine-tuning vs RAG"                 → M9 + M17 + M18
- "multimodal / vision + language"     → M13 + M14 + M17 (multi-modal RAG)
- "reasoning models / test-time RL"    → M9 (reasoning) + M21 (GRPO/DAPO)
"""

In [62]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────
import re
def llm_tree_search_with_expert(query: str, tree: list, model: str, expert_rules: str) -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
        Your task: identify which node IDs most likely contain the answer to the query.
        Think step-by-step about which sections are relevant.

        Query: {query}

        Document Tree:
        {json.dumps(compressed_tree, indent=2)}

        Expert Routing Rules (follow these carefully):
        {expert_rules}

        Reply ONLY in this exact JSON format:
        {{
            "thinking": "<your step-by-step reasoning>",
            "node_list": ["node_id1", "node_id2"]
        }}
    """

    response = model.invoke(prompt)
    content = response.content.strip()
    
    # Clean markdown formatting if present
    if content.startswith("```"):
        content = re.sub(r"^```(?:json)?\n?", "", content)
        content = re.sub(r"\n?```$", "", content)
        content = content.strip()
        
    try:
        parsed_response = json.loads(content, strict=False)
    except json.JSONDecodeError as e:
        # Fallback to try and extract json substring if LLM returned extra text
        match = re.search(r"\{.*\}", content, re.DOTALL)
        if match:
            parsed_response = json.loads(match.group(0), strict=False)
        else:
            raise e
            
    return parsed_response


In [64]:
# ── Test expert-guided retrieval ─────────────────────────────────────────────
query = "Details of RAG?"

print(f"🔍 Query: {query}\n")

# Without expert rules
print("── Without Expert Rules ──")
basic   = llm_tree_search(query, pageindex_tree, llm)
print("Nodes:", basic.get("node_list"))

print()

# With expert rules
print("── With Expert Rules ──")
guided  = llm_tree_search_with_expert(query, pageindex_tree, llm, FINANCIAL_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:300])

🔍 Query: Details of RAG?

── Without Expert Rules ──
Nodes: ['0002', '0015', '0016']

── With Expert Rules ──
Nodes: ['0002', '0006', '0012']
Reasoning: To identify which node IDs most likely contain the answer to the query 'Details of RAG?', I will follow these steps:

1. Identify the relevant module: Based on the query, I will route it to M17 RAG, as it deals with Retrieval-Augmented Generation (RAG).

2. Analyze the document tree: I will look for


---
## 💬 Chat API — Zero LLM Setup

**When to use this:**
- You don't want to manage OpenAI API calls yourself
- You want a quick Q&A interface over your document
- You're building a chat product and want PageIndex to handle everything

PageIndex provides its own LLM — you just pass a question and `doc_id`.


In [65]:
# ── Single question with Chat API ────────────────────────────────────────────
# No OpenAI key needed — PageIndex runs the LLM internally

question = "What are the key findings in this document?"

response = pi_client.chat_completions(
    messages=[{"role": "user", "content": question}],
    doc_id=doc_id
)

answer = response["choices"][0]["message"]["content"]
print("💬 Chat API Answer:")
print(answer)

💬 Chat API Answer:
Here are the **key findings** from this paper (PwC, arXiv:2511.18177):

---

## 🔍 Core Findings

### 1. Vector-Based Agentic RAG Outperforms Hierarchical Node-Based Systems
- **68% win rate** for vector-based agentic RAG over hierarchical node-based systems in LLM-as-a-judge pairwise comparisons.
- Comparable latency: **5.2 vs. 5.98 seconds** — so the quality gain comes with no meaningful speed penalty.
- The node-based system failed on 2 questions and gave 2 incorrect answers due to context window limitations; the vector-based system answered all questions successfully.
- The **key bottleneck** for node-based systems was at the table-of-contents selection stage, where the LLM struggled to identify relevant sections.

---

### 2. Cross-Encoder Reranking Dramatically Improves Retrieval Precision
- Baseline vector retrieval: **MRR@5 = 0.160**, Recall@5 = 0.50
- Optimal setting **(k_initial=10, k_final=5)**: **MRR@5 = 0.750**, Recall@5 = **1.00** — a **59% absolute impr

In [67]:
# ── Multi-turn conversation ───────────────────────────────────────────────────
# Keep the full message history for context across turns

conversation_history = []

def chat_with_doc(user_message: str, doc_id: str) -> str:
    """Chat with a document, maintaining conversation history."""
    global conversation_history
    
    conversation_history.append({"role": "user", "content": user_message})
    
    response = pi_client.chat_completions(
        messages=conversation_history,
        doc_id=doc_id
    )
    
    assistant_reply = response["choices"][0]["message"]["content"]
    conversation_history.append({"role": "assistant", "content": assistant_reply})
    
    return assistant_reply


# Simulate a 3-turn conversation
questions = [
    "WHat is RAG compared with?",
    "How is it efficient?",
    "what is the bottom line of this comparison"
]

for q in questions:
    print(f"\n👤 User: {q}")
    reply = chat_with_doc(q, doc_id)
    print(f"🤖 Assistant: {reply[:400]}...")
    print("-" * 55)


👤 User: WHat is RAG compared with?
🤖 Assistant: In this paper, **RAG (Retrieval-Augmented Generation)** is compared with **hierarchical node-based reasoning systems**. Specifically:

- **Vector-based Agentic RAG** — uses hybrid search (semantic vector search + BM25), metadata filtering, and dense embeddings to retrieve relevant chunks from a vector database.
- **Hierarchical Node-based Systems** — organize documents into a tree structure (like ...
-------------------------------------------------------

👤 User: How is it efficient?
🤖 Assistant: The paper addresses efficiency across **three dimensions** — latency, cost, and retrieval speed:

### ⏱️ Latency
- **Vector-based RAG** is slightly *faster* than the node-based system: **5.2 sec vs. 5.98 sec** end-to-end.
- **Small-to-big retrieval** adds only **+0.2 seconds** of latency, with near-zero extra cost ($0.000078/query) — making it highly efficient for the quality gain it provides.
- U...
--------------------------------------------

---
## 🛠️ Self-Hosted Open Source Option

**Use this when:**
- You don't want to send documents to any cloud
- You need full data privacy / on-prem deployment
- You want to inspect or customize the tree-building logic

The open-source repo at https://github.com/VectifyAI/PageIndex lets you run the entire pipeline locally using your own OpenAI key.

**What the CLI does:**
1. Reads your PDF
2. Detects existing Table of Contents (if any)
3. Uses GPT-4o to build the hierarchical tree
4. Saves a `document_name_pageindex.json` alongside your PDF


In [ ]:
# ── Clone the open-source repo ───────────────────────────────────────────────
!git clone https://github.com/VectifyAI/PageIndex.git
%cd PageIndex
!pip install -r requirements.txt

In [ ]:
# ── Create .env for self-hosted mode ─────────────────────────────────────────
# The local runner uses CHATGPT_API_KEY (not OPENAI_API_KEY)

import os
openai_key = os.getenv("OPENAI_API_KEY", "your_key_here")

with open(".env", "w") as f:
    f.write(f"CHATGPT_API_KEY={openai_key}\n")

print("✅ .env created for self-hosted mode")

In [ ]:
# ── Run PageIndex locally on a PDF ───────────────────────────────────────────
# Optional parameters you can customize:
#   --model                  OpenAI model (default: gpt-4o-2024-11-20)
#   --toc-check-pages        Pages to scan for existing TOC (default: 20)
#   --max-pages-per-node     Max pages per tree node (default: 10)
#   --if-add-node-summary    Include summaries in output (yes/no)

PDF_PATH = "/path/to/your/document.pdf"   # ← change this

!python run_pageindex.py \
    --pdf_path {PDF_PATH} \
    --model gpt-4o-2024-11-20 \
    --toc-check-pages 20 \
    --max-pages-per-node 10 \
    --if-add-node-summary yes

In [ ]:
# ── Load locally generated tree ──────────────────────────────────────────────
# Output is saved as: <your_pdf_name>_pageindex.json

import json

TREE_JSON_PATH = "/path/to/your/document_pageindex.json"  # ← change this

with open(TREE_JSON_PATH, "r") as f:
    local_tree = json.load(f)

print(f"🌲 Local tree loaded: {count_nodes(local_tree)} total nodes")
print_tree(local_tree)

In [ ]:
# ── Run the same RAG pipeline on the local tree ──────────────────────────────
# Everything from Sections 4–6 works identically with local trees

query  = "Summarize the executive summary section."
answer = vectorless_rag(query, local_tree)

---
## 📊 Vector RAG vs PageIndex — Side-by-Side

### Architecture Comparison

| Aspect | Traditional Vector RAG | PageIndex (Vectorless RAG) |
|--------|------------------------|---------------------------|
| **Document prep** | Chunk into fixed pieces | Build hierarchical tree |
| **Indexing** | Embed each chunk | LLM reads structure |
| **Storage** | Vector database | JSON file |
| **Query processing** | Embed query → ANN search | LLM reasons over tree |
| **What's retrieved** | Flat anonymous chunks | Named sections + page refs |
| **Explainability** | ❌ Opaque similarity score | ✅ Traceable reasoning |
| **Domain expertise** | ❌ Needs embedding fine-tune | ✅ Add rules to prompt |
| **Infrastructure** | Pinecone / FAISS / ChromaDB | No vector DB needed |
| **Best for** | Short, diverse documents | Long, structured documents |
| **FinanceBench accuracy** | ~80% | **98.7%** |

### When to use which

**Use Vector RAG when:**
- Documents are short and varied (FAQs, product descriptions)
- Semantic paraphrase matching is important  
- You need sub-second retrieval on millions of documents

**Use PageIndex when:**
- Documents are long and professionally structured (reports, manuals, legal docs)
- You need traceable, cited answers
- Domain expertise should guide retrieval
- You want to avoid vector DB infrastructure
